# 24 — A deep agent, end to end

This notebook assembles the pillars into one agent and runs a mini research-review: digest two sources, write an evidence-backed recommendation file. Contrast it with capstone 15: there, *code* owns the spine and calls agents at fixed points; here the *agent* owns a plan artifact and decides each step by re-reading it. The policy below is deterministic (the model-shaped boundary, as everywhere in this lab) and strictly **plan-driven** — it dispatches on `todos.next_pending()`, never on step counts — which is exactly what will make resume possible in notebook 25. Its fourth argument is the agent's live `Context`: the deterministic policies here don't need it, but it is the window a live model policy renders as its prompt — including any skills loaded via `load_skill` (notebook 28).

`DeepAgent.run()` literally calls notebook 01's `run_tool_loop` and returns the same `ToolLoopResult`; the deep disciplines all hook in at the tool boundary.

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd() / 'notebooks'))
from lab_core.deep import DeepAgent, SubAgent, VirtualFS, compose_system_prompt
from lab_core.patterns import Budget, CheckpointStore

fs = VirtualFS()
fs.write('/sources/doc-a.md', 'CLAIM: deterministic spines reduce variance.\n' + 'method detail. ' * 60)
fs.write('/sources/doc-b.md', 'CLAIM: objective verification beats self-approval.\n' + 'study detail. ' * 60)

def read_claims(task, context, files):
    path = task.split()[-1]
    text = files.read(path)
    context.add('tool:read_file', text[:120])
    claims = [line for line in text.splitlines() if line.startswith('CLAIM:')]
    return {'summary': '; '.join(claims) or 'no claims found', 'detail': text}

reader = SubAgent('reader', 'Extract the CLAIM lines from one source.', read_claims, fs)
print('sources ready:', fs.ls('/sources'))

In [ ]:
# The policy: ~20 lines, plan-driven, interleaving work with plan bookkeeping.
def policy(goal, todos, trace, context):
    if not todos.items:
        return {'tool': 'write_todos', 'input': {'add': [
            'survey sources', 'digest doc-a', 'digest doc-b', 'write recommendation']}}
    pending = todos.next_pending()
    if pending is None:
        return {'final': 'Recommendation written to /notes/recommendation.md'}
    index, last = todos.items.index(pending), (trace[-1] if trace else {})
    if pending.text == 'survey sources':
        if last.get('tool') == 'ls':
            return {'tool': 'write_todos', 'input': {'done': index, 'note': '2 sources'}}
        return {'tool': 'ls', 'input': {'prefix': '/sources'}}
    if pending.text.startswith('digest'):
        doc = pending.text.split()[-1]
        if last.get('tool') == 'task' and doc in str(last.get('input')):
            return {'tool': 'write_todos', 'input': {'done': index, 'note': 'delegated'}}
        return {'tool': 'task', 'input': {'agent': 'reader', 'task': f'extract claims from /sources/{doc}.md'}}
    if last.get('tool') == 'write_file':
        return {'tool': 'write_todos', 'input': {'done': index}}
    summaries = [t['observation'] for t in trace if t.get('tool') == 'task']
    text = 'Recommendation based on the digests:\n' + '\n'.join(f'- {s}' for s in summaries)
    return {'tool': 'write_file', 'input': {'path': '/notes/recommendation.md', 'text': text}}

In [ ]:
agent = DeepAgent(
    'reviewer',
    system_prompt=compose_system_prompt('You review sources and recommend with evidence.'),
    policy=policy,
    subagents=[reader],
    fs=fs,
    budget=Budget(max_steps=20, max_spend=1.0),
    checkpoints=CheckpointStore(),
)

result = agent.run('Review the sources and write a recommendation', run_id='review-1')
print('completed:', result.completed, '\n')
for step in result.trace:
    what = step.get('tool', 'final')
    detail = str(step.get('input', ''))[:46] if what != 'final' else str(step.get('output'))[:46]
    print(f"step {step['step']:>2}  {what:<12} {detail}")
print('\n' + agent.todos.render())
print('\nfiles:', agent.fs.ls())
print('budget used:', agent.budget.steps, 'of', agent.budget.max_steps, 'steps')

In [ ]:
# The context economics: where did the tokens go?
print('parent window entries:', [e.label for e in agent.context.entries])
print('parent window total:  ', agent.context.tokens(), 'tokens')
for report in agent.subagents['reader'].runs:
    detail_chars = len(agent.fs.read(report.detail_path))
    print(f'sub-agent window: {report.context.tokens():>4} tokens | '
          f'summary {len(report.summary)} chars crossed back | {detail_chars} chars stayed in files')
print()
print(agent.fs.read('/notes/recommendation.md'))

**Counterfactual.** Run this with notebook 23's flat pattern — every observation raw into one window — and the two source documents alone would put the window at several times this size and still growing, with the recommendation forced to compete with its own raw material for space. The deep agent's answer cites files instead of quoting them, and its window stayed flat across a ten-step run. Notebook 25: what happens when this run dies at step three.